# Buổi 1 — Nền tảng & Thống kê mô tả (Bài 1–3 của giáo trình ECNU)
**Mục tiêu:** (1) đọc dữ liệu, kiểm tra lỗi; (2) mô tả bằng số + biểu đồ; (3) hiểu vì sao *chỉ nhìn số trung bình là nguy hiểm*; (4) tương quan ≠ nhân quả.

Cách dùng: chạy lần lượt từng ô. Mỗi ô có phần **❓ Câu hỏi** — hãy tự trả lời trước khi xem đáp án của người hướng dẫn.

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

import numpy as np, pandas as pd

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' dùng cho cả 6 buổi (không phải dữ liệu thật)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)      # giờ tự học / tuần
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")           # hứng thú (ẩn)
    anxiety = rng.normal(0, 1, n) - 0.25 * interest                        # lo âu (ẩn)
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    # "bẫy" cố ý cài vào dữ liệu để buổi 1 phát hiện:
    d.loc[6, "study_hours"] = 48.0          # gõ nhầm 4.8 thành 48
    d.loc[[11, 57, 130], "h2"] = np.nan     # thiếu dữ liệu
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print(df.shape)

## 1. Nhìn dữ liệu trước khi tính bất cứ thứ gì
Tương ứng SPSS: *Data View / Variable View*, `Analyze > Descriptive Statistics > Frequencies`.

In [ ]:
df.head()

In [ ]:
df.info()                    # kiểu dữ liệu từng cột
print("\nSố ô thiếu theo cột:\n", df.isna().sum()[df.isna().sum() > 0])

### Thang đo (rất hay bị hỏi trong bài tập)
| Biến | Thang đo | Vì sao |
|---|---|---|
| `school`, `gender`, `method` | Định danh (nominal) | chỉ là nhãn, không có thứ tự |
| `h1..h3`, `a1..a3` | Thứ bậc (ordinal), thường coi như khoảng khi gộp thành thang điểm | Likert 1–5 |
| `study_hours`, `pretest`, `posttest`, `math` | Tỉ lệ / khoảng (scale) | có đơn vị, khoảng cách bằng nhau |

## 2. Bẫy #1 — giá trị ngoại lai (outlier) làm hỏng số trung bình

In [ ]:
print(df.study_hours.describe())
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
sns.histplot(df.study_hours, bins=30, ax=ax[0]); ax[0].set_title("Histogram study_hours")
sns.boxplot(x=df.study_hours, ax=ax[1]); ax[1].set_title("Boxplot")
plt.show()

In [ ]:
print("Dòng nghi vấn:"); print(df[df.study_hours > 20][["id", "study_hours", "math"]])
clean = df.copy(); clean.loc[clean.study_hours > 20, "study_hours"] = clean.loc[clean.study_hours > 20, "study_hours"] / 10   # 48 -> 4.8 (giả định gõ nhầm dấu phẩy)
cmp = pd.DataFrame({"Trước khi sửa": df.study_hours.agg(["mean", "median", "std"]),
                    "Sau khi sửa":   clean.study_hours.agg(["mean", "median", "std"])})
cmp

**❓ Câu hỏi**
1. Trung bình thay đổi bao nhiêu? Trung vị thay đổi bao nhiêu? Vì sao trung vị "bền" hơn?
2. Có được **tự ý** sửa 48 → 4.8 không? Điều gì cần làm trước khi sửa (hỏi người thu thập, xem phiếu gốc, ghi vào nhật ký xử lý)?
3. Nếu là ngoại lai *thật* (một em học 48 giờ/tuần), bạn xử lý khác đi thế nào?

## 3. Bẫy #2 — cùng trung bình, câu chuyện hoàn toàn khác
SPSS: `Analyze > Descriptive Statistics > Explore` (kèm Boxplot).

In [ ]:
clean = clean.assign(h_mean=clean[["h1", "h2", "h3"]].mean(axis=1))
print(clean.groupby("school").math.agg(["count", "mean", "median", "std"]))
sns.boxplot(data=clean, x="school", y="math"); sns.stripplot(data=clean, x="school", y="math", color="k", alpha=.25, size=3)
plt.title("Điểm toán theo trường: xem cả phân bố, không chỉ trung bình"); plt.show()

### Bẫy #3 — Bộ tứ Anscombe: 4 tập dữ liệu, cùng mean, cùng phương sai, cùng r ≈ 0.816, cùng đường hồi quy
Chỉ có **biểu đồ** mới phân biệt được.

In [ ]:
x = [10,8,13,9,11,14,6,4,12,7,5]
ys = {"I":[8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68],
      "II":[9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74],
      "III":[7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73]}
x4 = [8,8,8,8,8,8,8,19,8,8,8]; y4 = [6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89]
sets = {k: (x, v) for k, v in ys.items()}; sets["IV"] = (x4, y4)
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, (k, (a, b)) in zip(axes, sets.items()):
    r = np.corrcoef(a, b)[0, 1]; sns.regplot(x=a, y=b, ax=ax, ci=None); ax.set_title(f"{k}: r={r:.3f}, mean_y={np.mean(b):.2f}")
plt.show()

## 4. Tương quan (Bài 2 giáo trình) — Pearson vs Spearman
SPSS: `Analyze > Correlate > Bivariate`.

In [ ]:
cols = ["study_hours", "pretest", "posttest", "math"]
print(clean[cols].corr(method="pearson").round(2), "\n")
print(clean[cols].corr(method="spearman").round(2))
sns.heatmap(clean[cols].corr(), annot=True, cmap="vlag", vmin=-1, vmax=1); plt.show()

In [ ]:
r, p = stats.pearsonr(clean.study_hours, clean.math)
print(f"r = {r:.3f}, r² = {r**2:.3f}, p = {p:.2g}  → giờ tự học 'giải thích' ~{r**2:.0%} phương sai điểm toán")
sns.regplot(data=clean, x="study_hours", y="math", scatter_kws={"alpha": .4}); plt.show()

### Bẫy #4 — chọn mẫu hẹp làm r "biến mất" (range restriction)

In [ ]:
top = clean[clean.math > clean.math.quantile(.6)]
print("r toàn mẫu:", round(clean.study_hours.corr(clean.math), 3), "| r chỉ nhóm điểm cao:", round(top.study_hours.corr(top.math), 3))

**❓ Câu hỏi:** Nếu chỉ khảo sát học sinh trường chuyên (điểm cao), bạn sẽ kết luận gì về mối quan hệ giờ học – điểm? Vì sao kết luận đó sai cho học sinh nói chung?

## 5. Bài tập về nhà
1. Lặp lại phần 2–4 cho biến `posttest`. Có ngoại lai nào không? (gợi ý: quy tắc 1.5×IQR)
2. Viết 5 câu nhận xét (như trong báo cáo) về mô tả `math` theo 3 trường. Chỉ dùng số liệu bạn vừa tính.
3. (Thử thách) Tính tay hệ số tương quan Pearson từ công thức `Σ(x-x̄)(y-ȳ) / √(Σ(x-x̄)²·Σ(y-ȳ)²)` và so với `pearsonr`.